# 💳 Credit Card Fraud Detection — End-to-End Machine Learning Project
### A Professional, Beginner-Friendly Guide using Logistic Regression

---
**Author:** Senior Data Scientist / ML Engineer  
**Dataset:** `credit_card_fraud_10k.csv` (10,000 transactions)  
**Goal:** Build a production-quality fraud detection model with full EDA, feature engineering, and model evaluation.

---
## 📚 Table of Contents
1. [Phase 1 — Data Understanding](#phase1)
2. [Phase 2 — Data Cleaning](#phase2)
3. [Phase 3 — Exploratory Data Analysis (EDA)](#phase3)
4. [Phase 4 — Data Visualization](#phase4)
5. [Phase 5 — Feature Engineering & Preprocessing](#phase5)
6. [Phase 6 — Machine Learning Model](#phase6)
7. [Phase 7 — Model Improvement](#phase7)
8. [Phase 8 — Final Insights & Conclusion](#phase8)


---
## 🔍 Phase 1 — Data Understanding <a id='phase1'></a>

### What are we doing here?
Before writing a single line of ML code, we **must understand our data deeply**.  
Think of it like a doctor reviewing patient history before prescribing medicine.

### Business Problem 🎯
Credit card fraud costs the global economy **billions of dollars annually**.  
Our job: build a model that can **automatically flag suspicious transactions**  
so the bank can block or review them before money is lost.

### The Target Variable
- `is_fraud` = **1** → This transaction IS fraudulent (we want to catch these!)
- `is_fraud` = **0** → This transaction is legitimate

### ⚠️ Key Challenge
Only **~1.5%** of transactions are fraud. This is called a **class imbalance problem**.  
We'll handle this carefully — accuracy alone will be misleading here.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 1: Install / Import Libraries
# ─────────────────────────────────────────────────────────────────

# All libraries below come pre-installed in Google Colab.
# If any are missing, uncomment the pip install lines.

# !pip install scikit-learn imbalanced-learn seaborn matplotlib pandas numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Sklearn — the main ML toolkit
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.pipeline import Pipeline

# Imbalanced-learn — handles class imbalance
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')  # Suppress non-critical warnings

# ── Global Plot Style ──────────────────────────────────────────
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_theme(style='whitegrid', palette='husl')

print("✅ All libraries imported successfully!")
print(f"   pandas    : {pd.__version__}")
print(f"   numpy     : {np.__version__}")
print(f"   seaborn   : {sns.__version__}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2: Load the Dataset
# ─────────────────────────────────────────────────────────────────

# 📌 HOW TO UPLOAD IN COLAB:
#    Sidebar → Files icon → Upload → select credit_card_fraud_10k.csv
#    OR run: from google.colab import files; files.upload()

df = pd.read_csv('credit_card_fraud_10k.csv')

print("✅ Dataset loaded successfully!")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3: First Look at the Data
# ─────────────────────────────────────────────────────────────────

print("="*60)
print(" FIRST 5 ROWS (HEAD)")
print("="*60)
display(df.head())

print("\n" + "="*60)
print(" LAST 5 ROWS (TAIL)")
print("="*60)
display(df.tail())


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4: Column-by-Column Explanation
# ─────────────────────────────────────────────────────────────────

column_guide = {
    'transaction_id'      : 'Unique ID for each transaction. NOT a feature — just an identifier.',
    'amount'              : 'Dollar value of the transaction. Fraudsters often test with tiny amounts or go very large.',
    'transaction_hour'    : 'Hour of day (0–23). Fraudulent activity may spike at unusual hours (e.g., 2 AM).',
    'merchant_category'   : 'Type of store/merchant: Food, Travel, Electronics, Grocery, Clothing.',
    'foreign_transaction' : '1 = transaction occurred in a foreign country. Often a red flag for fraud.',
    'location_mismatch'   : '1 = billing address doesn't match transaction location. Strong fraud signal.',
    'device_trust_score'  : 'Score 25–99 rating how trusted the device is. Lower = less trusted device.',
    'velocity_last_24h'   : 'Number of transactions made in the last 24 hours. High velocity = suspicious.',
    'cardholder_age'      : 'Age of the credit card owner (18–69).',
    'is_fraud'            : '🎯 TARGET: 1 = Fraud, 0 = Legitimate. This is what we predict.'
}

print("📖 COLUMN GUIDE — What Each Feature Means:")
print("="*70)
for col, desc in column_guide.items():
    print(f"  {col:<25} → {desc}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5: Data Types & Summary Statistics
# ─────────────────────────────────────────────────────────────────

print("DATA TYPES:")
print("-"*40)
print(df.dtypes)

print("\nSUMMARY STATISTICS (Numerical Features):")
print("-"*60)
display(df.describe().round(2))

print("\nSUMMARY STATISTICS (Categorical Features):")
print("-"*60)
display(df.describe(include='object'))


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6: Target Variable Analysis
# ─────────────────────────────────────────────────────────────────

fraud_counts = df['is_fraud'].value_counts()
fraud_pct    = df['is_fraud'].value_counts(normalize=True) * 100

print("🎯 TARGET VARIABLE: is_fraud")
print("="*40)
print(f"  Legitimate (0) : {fraud_counts[0]:,}  ({fraud_pct[0]:.2f}%)")
print(f"  Fraudulent (1) : {fraud_counts[1]:,}   ({fraud_pct[1]:.2f}%)")

print("\n⚠️  CLASS IMBALANCE ALERT:")
print(f"   The dataset has {fraud_pct[1]:.2f}% fraud cases.")
print("   A naive model predicting 'all legitimate' would be")
print(f"   {fraud_pct[0]:.2f}% 'accurate' — but USELESS for fraud detection!")
print("   We MUST use Recall & F1-score, not just Accuracy.")


### 📝 Phase 1 Summary
| Property | Value |
|---|---|
| Total Rows | 10,000 |
| Total Columns | 10 |
| Target | `is_fraud` (binary: 0/1) |
| Fraud Rate | ~1.51% |
| Categorical Features | `merchant_category` |
| Binary Flag Features | `foreign_transaction`, `location_mismatch` |
| Numerical Features | `amount`, `transaction_hour`, `device_trust_score`, `velocity_last_24h`, `cardholder_age` |
| Class Imbalance | ⚠️ Severe — 98.5% legitimate vs 1.5% fraud |

**Key Insight:** With only 1.51% fraud, this is a heavily imbalanced dataset.  
Accuracy is not our metric — **Recall** (catching actual frauds) matters most.


---
## 🧹 Phase 2 — Data Cleaning <a id='phase2'></a>

### What are we doing here?
Garbage in = garbage out. Even the best ML model fails on dirty data.  
We check for: missing values, duplicates, wrong data types, and outliers.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 7: Missing Values Check
# ─────────────────────────────────────────────────────────────────

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_report = pd.DataFrame({
    'Missing Count' : missing,
    'Missing %'     : missing_pct.round(2)
})

print("🔍 MISSING VALUES REPORT:")
print("="*40)
print(missing_report)
print(f"\n✅ Total missing values: {missing.sum()}")

if missing.sum() == 0:
    print("   No missing values found. No imputation needed.")
else:
    print("   Action required — see imputation logic below.")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 8: Duplicate Rows Check
# ─────────────────────────────────────────────────────────────────

dup_count = df.duplicated().sum()
print(f"🔍 Duplicate rows found: {dup_count}")

if dup_count > 0:
    print(f"   Removing {dup_count} duplicate rows...")
    df = df.drop_duplicates()
    print(f"   Remaining rows: {len(df):,}")
else:
    print("   ✅ No duplicates found. Dataset is clean.")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 9: Data Type Validation
# ─────────────────────────────────────────────────────────────────

print("🔍 CHECKING DATA TYPE CORRECTNESS:")
print("-"*50)

# Check that binary flags are truly 0/1
for col in ['foreign_transaction', 'location_mismatch', 'is_fraud']:
    unique_vals = sorted(df[col].unique())
    expected    = [0, 1]
    status      = '✅' if unique_vals == expected else '❌ UNEXPECTED VALUES'
    print(f"  {col:<25} values: {unique_vals}  {status}")

# Check merchant_category
print(f"\n  merchant_category unique values: {df['merchant_category'].unique().tolist()}")
print("  ✅ All expected categories present")

# Check numeric ranges
print("\n🔍 NUMERIC RANGE CHECK:")
print(f"  transaction_hour  : {df['transaction_hour'].min()} – {df['transaction_hour'].max()} (expected 0–23)")
print(f"  device_trust_score: {df['device_trust_score'].min()} – {df['device_trust_score'].max()} (expected 25–99)")
print(f"  velocity_last_24h : {df['velocity_last_24h'].min()} – {df['velocity_last_24h'].max()} (expected 0–9)")
print(f"  cardholder_age    : {df['cardholder_age'].min()} – {df['cardholder_age'].max()} (expected 18–69)")
print(f"  amount            : {df['amount'].min():.2f} – {df['amount'].max():.2f}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 10: Outlier Detection (IQR Method)
# ─────────────────────────────────────────────────────────────────

# We focus on 'amount' since it's the most likely to have extreme values.
# For fraud detection, high-amount transactions are MEANINGFUL outliers,
# so we DO NOT remove them — we just understand them.

numerical_cols = ['amount', 'device_trust_score', 'velocity_last_24h', 'cardholder_age']

print("🔍 OUTLIER DETECTION USING IQR METHOD:")
print("-"*55)
print(f"  {'Feature':<25} {'Outliers':>10}  {'% of Data':>10}")
print("-"*55)

for col in numerical_cols:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = ((df[col] < lower) | (df[col] > upper)).sum()
    pct    = n_out / len(df) * 100
    print(f"  {col:<25} {n_out:>10}  {pct:>9.2f}%")

print("""
⚠️  OUTLIER STRATEGY:
   'amount' outliers = high-value transactions. In fraud detection,
   these are IMPORTANT signals, NOT noise. We keep them.
   Removing them would delete valuable fraud patterns.
   
   Instead, we will apply StandardScaler in Phase 5 to
   normalize the scale without losing the signal.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 11: Drop Unnecessary Columns
# ─────────────────────────────────────────────────────────────────

# 'transaction_id' is just an index — not a feature.
# Keeping it would confuse the model (it's sequential, not predictive).

print("Dropping 'transaction_id' — it's an identifier, not a feature.")
df = df.drop(columns=['transaction_id'])
print(f"✅ Remaining columns: {df.columns.tolist()}")
print(f"   Dataset shape after cleaning: {df.shape}")


### 📝 Phase 2 Summary
| Check | Result |
|---|---|
| Missing Values | ✅ None found |
| Duplicate Rows | ✅ None found |
| Data Types | ✅ All correct |
| Outliers | ⚠️ Present in `amount` — kept intentionally |
| Columns Dropped | `transaction_id` (identifier, not predictive) |

**Decision:** Dataset is already clean. The only action was dropping the ID column.


---
## 📊 Phase 3 — Exploratory Data Analysis (EDA) <a id='phase3'></a>

### What are we doing here?
EDA is about telling the **data's story before we model it**.  
We want to understand patterns, distributions, and relationships —  
especially how features differ between fraud and legitimate transactions.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 12: Class Distribution
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Bar Chart ──────────────────────────────────────────────────
labels = ['Legitimate (0)', 'Fraud (1)']
counts = df['is_fraud'].value_counts().values
colors = ['#2196F3', '#F44336']

bars = axes[0].bar(labels, counts, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Transaction Class Distribution', fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}', ha='center', fontweight='bold')

# ── Pie Chart ──────────────────────────────────────────────────
axes[1].pie(counts, labels=labels, autopct='%1.2f%%',
            colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'black', 'linewidth': 1.2})
axes[1].set_title('Fraud vs Legitimate (Proportion)', fontweight='bold')

plt.suptitle('Class Imbalance Overview', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHT:
   Only 1.51% of transactions are fraudulent (151 out of 10,000).
   This is called CLASS IMBALANCE.
   → A model that says 'all legit' gets 98.49% accuracy but 0% fraud detection.
   → We MUST use techniques like SMOTE and evaluate with Recall & F1.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 13: Numerical Feature Distributions
# ─────────────────────────────────────────────────────────────────

# Why? — To understand the shape of each feature.
# Skewed distributions can hurt Logistic Regression.

numerical_cols = ['amount', 'transaction_hour', 'device_trust_score',
                  'velocity_last_24h', 'cardholder_age']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=30, color='#42A5F5', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Distribution of {col}', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    
    # Add mean line
    mean_val = df[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.5,
                    label=f'Mean: {mean_val:.1f}')
    axes[i].legend(fontsize=9)

axes[5].axis('off')  # Hide unused subplot

plt.suptitle('Numerical Feature Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHTS:
   • 'amount' is right-skewed (most transactions are small; a few are very large).
     This is typical of real fraud datasets — StandardScaler will help.
   • 'transaction_hour' appears roughly uniform (transactions happen all hours).
   • 'device_trust_score' is roughly uniform (25–99 range).
   • 'velocity_last_24h' is right-skewed — most users make 0–2 transactions/day.
   • 'cardholder_age' appears roughly uniform between 18–69.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 14: Fraud vs Legitimate — Amount Distribution
# ─────────────────────────────────────────────────────────────────

# One of the MOST IMPORTANT plots for fraud detection.
# Do fraudsters spend differently than legitimate users?

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Overlapping Histogram ──────────────────────────────────────
legit = df[df['is_fraud'] == 0]['amount']
fraud = df[df['is_fraud'] == 1]['amount']

axes[0].hist(legit, bins=50, alpha=0.6, color='#2196F3', label='Legitimate', edgecolor='black')
axes[0].hist(fraud, bins=50, alpha=0.7, color='#F44336', label='Fraud',      edgecolor='black')
axes[0].set_title('Transaction Amount: Fraud vs Legitimate', fontweight='bold')
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# ── Box Plot ──────────────────────────────────────────────────
df_plot = df.copy()
df_plot['Label'] = df_plot['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})
axes[1].boxplot([legit, fraud], labels=['Legitimate', 'Fraud'],
                patch_artist=True,
                boxprops=dict(facecolor='#90CAF9'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Amount Distribution (Box Plot)', fontweight='bold')
axes[1].set_ylabel('Amount ($)')

plt.tight_layout()
plt.savefig('amount_fraud_vs_legit.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHTS:
   Median legitimate amount: ${:.2f}
   Median fraud amount     : ${:.2f}
   → Fraudulent transactions tend to involve different amounts.
   'amount' is likely a useful predictive feature.
""".format(legit.median(), fraud.median()))


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 15: Fraud Rate by Categorical & Binary Features
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Merchant Category ──────────────────────────────────────────
fraud_by_merchant = df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False) * 100
bars = axes[0].bar(fraud_by_merchant.index, fraud_by_merchant.values,
                   color=sns.color_palette('Reds_r', len(fraud_by_merchant)), edgecolor='black')
axes[0].set_title('Fraud Rate by Merchant Category', fontweight='bold')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_xlabel('Merchant Category')
axes[0].tick_params(axis='x', rotation=15)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}%', ha='center', fontsize=9)

# ── Foreign Transaction ────────────────────────────────────────
fraud_foreign = df.groupby('foreign_transaction')['is_fraud'].mean() * 100
labels_f = ['Domestic (0)', 'Foreign (1)']
bars2 = axes[1].bar(labels_f, fraud_foreign.values, color=['#42A5F5', '#EF5350'], edgecolor='black')
axes[1].set_title('Fraud Rate: Foreign vs Domestic', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}%', ha='center', fontweight='bold')

# ── Location Mismatch ──────────────────────────────────────────
fraud_loc = df.groupby('location_mismatch')['is_fraud'].mean() * 100
labels_l = ['No Mismatch (0)', 'Mismatch (1)']
bars3 = axes[2].bar(labels_l, fraud_loc.values, color=['#66BB6A', '#FF7043'], edgecolor='black')
axes[2].set_title('Fraud Rate: Location Mismatch', fontweight='bold')
axes[2].set_ylabel('Fraud Rate (%)')
for bar in bars3:
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}%', ha='center', fontweight='bold')

plt.suptitle('Fraud Rate Analysis by Key Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fraud_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHTS:
   • Grocery has the highest fraud rate, Electronics the lowest.
   • Foreign transactions are significantly more likely to be fraudulent.
   • Location mismatch transactions have a much higher fraud rate.
   → 'foreign_transaction' and 'location_mismatch' are strong fraud signals!
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 16: Transaction Hour — Fraud Heatmap
# ─────────────────────────────────────────────────────────────────

# Do fraudsters operate at specific times of day?

hourly = df.groupby('transaction_hour')['is_fraud'].agg(['sum', 'count'])
hourly['fraud_rate'] = (hourly['sum'] / hourly['count']) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Fraud Rate by Hour ─────────────────────────────────────────
axes[0].plot(hourly.index, hourly['fraud_rate'], marker='o', color='#E53935', linewidth=2)
axes[0].fill_between(hourly.index, hourly['fraud_rate'], alpha=0.2, color='#E53935')
axes[0].set_title('Fraud Rate by Transaction Hour', fontweight='bold')
axes[0].set_xlabel('Hour of Day (0–23)')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_xticks(range(0, 24))

# ── Transaction Volume by Hour ────────────────────────────────
axes[1].bar(hourly.index, hourly['count'], color='#42A5F5', edgecolor='black', alpha=0.8)
axes[1].set_title('Transaction Volume by Hour', fontweight='bold')
axes[1].set_xlabel('Hour of Day (0–23)')
axes[1].set_ylabel('Number of Transactions')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig('fraud_by_hour.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHT:
   Transaction volume is relatively uniform across all hours.
   However, fraud rate fluctuates by hour — some late-night hours
   may show elevated fraud rates. This feature may still hold
   predictive value when combined with others.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 17: Velocity Analysis — Transactions in Last 24h
# ─────────────────────────────────────────────────────────────────

# High velocity = many transactions in a short time = suspicious pattern

velocity_fraud = df.groupby('velocity_last_24h')['is_fraud'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Fraud Rate by Velocity ─────────────────────────────────────
axes[0].bar(velocity_fraud.index, velocity_fraud.values,
            color=sns.color_palette('Reds', len(velocity_fraud)), edgecolor='black')
axes[0].set_title('Fraud Rate by Transaction Velocity (Last 24h)', fontweight='bold')
axes[0].set_xlabel('Number of Transactions in Last 24 Hours')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].set_xticks(velocity_fraud.index)

# ── Count Distribution ─────────────────────────────────────────
velocity_counts = df['velocity_last_24h'].value_counts().sort_index()
axes[1].bar(velocity_counts.index, velocity_counts.values,
            color='#42A5F5', edgecolor='black')
axes[1].set_title('User Velocity Distribution (Last 24h)', fontweight='bold')
axes[1].set_xlabel('Number of Transactions in Last 24 Hours')
axes[1].set_ylabel('Number of Users')
axes[1].set_xticks(velocity_counts.index)

plt.tight_layout()
plt.savefig('velocity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHT:
   Higher velocity (more transactions in 24h) correlates with
   higher fraud rates. This is a valuable fraud signal —
   fraudsters often make many quick transactions before being caught.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 18: Device Trust Score — Fraud vs Legitimate
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── KDE Plot ──────────────────────────────────────────────────
df[df['is_fraud'] == 0]['device_trust_score'].plot(kind='kde', ax=axes[0],
    color='#2196F3', label='Legitimate', linewidth=2)
df[df['is_fraud'] == 1]['device_trust_score'].plot(kind='kde', ax=axes[0],
    color='#F44336', label='Fraud', linewidth=2, linestyle='--')
axes[0].set_title('Device Trust Score: Fraud vs Legitimate', fontweight='bold')
axes[0].set_xlabel('Device Trust Score')
axes[0].legend()

# ── Violin Plot ───────────────────────────────────────────────
import matplotlib.patches as mpatches
data_legit = df[df['is_fraud'] == 0]['device_trust_score']
data_fraud = df[df['is_fraud'] == 1]['device_trust_score']
parts = axes[1].violinplot([data_legit, data_fraud], positions=[1, 2], showmedians=True)

# Color the violins
parts['bodies'][0].set_facecolor('#2196F3')
parts['bodies'][1].set_facecolor('#F44336')
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['Legitimate', 'Fraud'])
axes[1].set_title('Device Trust Score Distribution (Violin)', fontweight='bold')
axes[1].set_ylabel('Device Trust Score')

plt.tight_layout()
plt.savefig('device_trust_score.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHT:
   If fraudulent transactions tend to occur on lower-trust devices,
   'device_trust_score' is a strong predictive feature.
   The violin plot shows the full distribution shape for each class.
""")


### 📝 Phase 3 & 4 Summary — Key EDA Findings

| Feature | Fraud Signal Strength | Key Finding |
|---|---|---|
| `foreign_transaction` | 🔴 HIGH | Foreign transactions are much more likely to be fraud |
| `location_mismatch` | 🔴 HIGH | Mismatch strongly associated with fraud |
| `velocity_last_24h` | 🟠 MEDIUM-HIGH | Higher velocity = higher fraud rate |
| `device_trust_score` | 🟠 MEDIUM | Lower score correlates with fraud |
| `amount` | 🟡 MEDIUM | Distribution differs between fraud/legit |
| `merchant_category` | 🟡 MEDIUM | Grocery has highest fraud rate |
| `transaction_hour` | 🟢 LOW-MEDIUM | Some hour-based patterns exist |
| `cardholder_age` | 🟢 LOW | Weak signal, but included |


---
## 📈 Phase 4 — Advanced Visualizations <a id='phase4'></a>


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 19: Correlation Heatmap
# ─────────────────────────────────────────────────────────────────

# Why? — Shows which features are correlated with each other
# and with the target (is_fraud). Helps detect multicollinearity.

# First, encode merchant_category temporarily for correlation
df_corr = df.copy()
le_temp = LabelEncoder()
df_corr['merchant_category_enc'] = le_temp.fit_transform(df_corr['merchant_category'])
df_corr = df_corr.drop(columns=['merchant_category'])

corr_matrix = df_corr.corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
heatmap = sns.heatmap(
    corr_matrix, 
    mask=mask,
    annot=True, 
    fmt='.3f', 
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Matrix\n(Focus on is_fraud row)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Feature correlation with target
print("\n📊 CORRELATION WITH TARGET (is_fraud) — Sorted:")
target_corr = corr_matrix['is_fraud'].drop('is_fraud').sort_values(key=abs, ascending=False)
for feat, corr_val in target_corr.items():
    bar = '█' * int(abs(corr_val) * 30)
    direction = '+' if corr_val > 0 else '-'
    print(f"  {feat:<30} {direction}{abs(corr_val):.4f}  {bar}")

print("""
💡 INTERPRETATION:
   • Positive correlation → higher values → more likely fraud
   • Negative correlation → higher values → less likely fraud
   • Values close to 0 → weak linear relationship
   Note: Logistic Regression can still use weak-correlation features
   in combination for improved prediction.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 20: Pair Plot — Key Features
# ─────────────────────────────────────────────────────────────────

# Pair plots show relationships between features, colored by class.
# Useful for spotting clusters that separate fraud from legit.

# We use a subset of features to keep the plot readable.
pair_features = ['amount', 'device_trust_score', 'velocity_last_24h',
                 'foreign_transaction', 'location_mismatch', 'is_fraud']

df_pair = df[pair_features].copy()
df_pair['Fraud'] = df_pair['is_fraud'].map({0: 'Legitimate', 1: 'Fraud'})

pair_grid = sns.pairplot(
    df_pair.drop(columns=['is_fraud']),
    hue='Fraud',
    palette={'Legitimate': '#2196F3', 'Fraud': '#F44336'},
    plot_kws={'alpha': 0.4, 's': 20},
    diag_kind='kde',
    corner=True
)
pair_grid.fig.suptitle('Pair Plot: Key Features (Fraud vs Legitimate)', 
                         fontsize=14, fontweight='bold', y=1.01)
plt.savefig('pair_plot.png', dpi=120, bbox_inches='tight')
plt.show()

print("""
💡 INSIGHT:
   The pair plot reveals whether fraud cases cluster distinctly
   from legitimate ones. Overlapping clusters = harder classification.
   Clear separation = easier for the model to learn boundaries.
""")


---
## ⚙️ Phase 5 — Feature Engineering & Preprocessing <a id='phase5'></a>

### What are we doing here?
Machine learning models can't work with raw data as-is.  
We need to:
1. **Encode** categorical text variables into numbers
2. **Scale** numerical features to a common range
3. **Handle class imbalance** using SMOTE (Synthetic Minority Over-sampling)
4. **Split** into train/test sets properly


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 21: Feature Engineering
# ─────────────────────────────────────────────────────────────────

# ── New Feature 1: Risk Score ──────────────────────────────────
# Combine the two strongest fraud signals into a single risk score.
# If foreign AND location mismatch → double red flag.
df['combined_risk_flag'] = (
    (df['foreign_transaction'] == 1) & (df['location_mismatch'] == 1)
).astype(int)

# ── New Feature 2: High Velocity Flag ─────────────────────────
# Mark users with unusually high transaction frequency.
df['high_velocity'] = (df['velocity_last_24h'] >= 5).astype(int)

# ── New Feature 3: Low Trust Device ───────────────────────────
# Devices with trust score below 50 are flagged.
df['low_trust_device'] = (df['device_trust_score'] < 50).astype(int)

# ── New Feature 4: Night Transaction ──────────────────────────
# Transactions between 11 PM – 5 AM are flagged.
df['night_transaction'] = df['transaction_hour'].apply(
    lambda h: 1 if (h >= 23 or h <= 5) else 0
)

print("✅ New features created:")
print("  combined_risk_flag : foreign + location mismatch combined")
print("  high_velocity      : 5+ transactions in 24h")
print("  low_trust_device   : device trust score < 50")
print("  night_transaction  : transaction between 11 PM – 5 AM")

print("\nFraud rates for new features:")
for feat in ['combined_risk_flag', 'high_velocity', 'low_trust_device', 'night_transaction']:
    rate = df[df[feat] == 1]['is_fraud'].mean() * 100
    print(f"  {feat:<25}: {rate:.2f}% fraud rate when flag=1")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 22: Encode Categorical Variable
# ─────────────────────────────────────────────────────────────────

# 'merchant_category' has 5 categories: Food, Travel, Grocery, Clothing, Electronics
# We use One-Hot Encoding — better than Label Encoding for non-ordinal categories.
# Label Encoding would imply Food > Travel > Grocery (false ordering).

df_model = pd.get_dummies(df, columns=['merchant_category'], drop_first=True)

print("✅ One-Hot Encoding applied to 'merchant_category'")
print(f"   Original columns : {df.shape[1]}")
print(f"   After encoding   : {df_model.shape[1]}")
print(f"   New columns      : {[c for c in df_model.columns if 'merchant_category' in c]}")
print(f"\nAll columns after encoding:")
for col in df_model.columns:
    print(f"   {col}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 23: Define Features & Target
# ─────────────────────────────────────────────────────────────────

TARGET = 'is_fraud'

# All columns except the target
FEATURES = [col for col in df_model.columns if col != TARGET]

X = df_model[FEATURES]
y = df_model[TARGET]

print(f"✅ Features (X) shape : {X.shape}")
print(f"   Target  (y) shape : {y.shape}")
print(f"   Number of features: {len(FEATURES)}")
print(f"\n   Features used:")
for f in FEATURES:
    print(f"     • {f}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 24: Train-Test Split
# ─────────────────────────────────────────────────────────────────

# stratify=y ensures BOTH train and test sets have the same
# fraud proportion (~1.5%). Without this, by chance all fraud
# cases could end up in training or all in test.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # reproducible split
    stratify=y           # ← CRITICAL for imbalanced datasets
)

print("✅ Train-Test Split (80/20) with stratification:")
print(f"   Training set   : {X_train.shape[0]:,} rows")
print(f"   Testing set    : {X_test.shape[0]:,} rows")
print(f"   Train fraud %  : {y_train.mean()*100:.2f}%")
print(f"   Test  fraud %  : {y_test.mean()*100:.2f}%")
print("   ✅ Fraud proportion maintained in both sets (stratify worked)")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 25: Feature Scaling
# ─────────────────────────────────────────────────────────────────

# Logistic Regression is sensitive to feature scale.
# 'amount' ranges 0–1471, 'device_trust_score' 25–99.
# Without scaling, 'amount' would dominate the model unfairly.
#
# StandardScaler: transforms each feature to mean=0, std=1
# IMPORTANT: Fit ONLY on training data, then transform both sets.
# (Fitting on test data = data leakage = cheating!)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # ONLY transform on test

print("✅ StandardScaler applied:")
print("   Fit on training set only (no data leakage).")
print(f"   Scaled training mean (first 3 features): {X_train_scaled[:, :3].mean(axis=0).round(4)}")
print(f"   Scaled training std  (first 3 features): {X_train_scaled[:, :3].std(axis=0).round(4)}")
print("   → means ≈ 0, std ≈ 1 (as expected after scaling)")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 26: Handle Class Imbalance with SMOTE
# ─────────────────────────────────────────────────────────────────

# SMOTE = Synthetic Minority Over-sampling Technique
#
# How it works:
# 1. Takes an existing fraud sample
# 2. Finds its nearest fraud neighbors
# 3. Creates a NEW synthetic fraud sample between them
# 4. This gives the model more fraud examples to learn from
#
# CRITICAL: Apply SMOTE ONLY to training data.
# Never apply to test data — that would be unrealistic.

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("✅ SMOTE Applied to Training Set:")
print(f"   Before SMOTE: {len(y_train):,} samples  (Fraud: {y_train.sum()}, Legit: {(y_train==0).sum()})")
print(f"   After  SMOTE: {len(y_train_res):,} samples  (Fraud: {y_train_res.sum()}, Legit: {(y_train_res==0).sum()})")
print(f"   Training fraud % after SMOTE: {y_train_res.mean()*100:.2f}%")
print("\n   Test set is UNCHANGED (real-world distribution preserved):")
print(f"   Test fraud %: {y_test.mean()*100:.2f}%")


### 📝 Phase 5 Summary
| Step | Action | Reason |
|---|---|---|
| Feature Engineering | Created 4 new risk flags | Domain knowledge boosts model |
| Encoding | One-Hot on `merchant_category` | Avoids false ordinal relationships |
| Split | 80/20 with stratify | Maintains fraud ratio in both sets |
| Scaling | StandardScaler (fit on train only) | Logistic Regression needs equal scale |
| Imbalance | SMOTE on training set only | Gives model more fraud examples |


---
## 🤖 Phase 6 — Machine Learning Model (Logistic Regression) <a id='phase6'></a>

### What is Logistic Regression?
Despite the name, it's a **classification** algorithm (not regression).  
It estimates the **probability** that a transaction is fraud (between 0 and 1).

**Formula:**  P(fraud) = 1 / (1 + e^(-z))  where z = w₁x₁ + w₂x₂ + ... + wₙxₙ

If P(fraud) > 0.5 → predict fraud  
If P(fraud) ≤ 0.5 → predict legitimate

### Why Logistic Regression First?
1. **Interpretable** — coefficients tell us exactly which features drive fraud
2. **Fast** — trains in milliseconds on 10k rows
3. **Baseline** — gives us a benchmark to compare against complex models
4. **Probabilistic** — outputs probabilities, not just 0/1 labels (useful for threshold tuning)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 27: Train Baseline Logistic Regression (No SMOTE)
# ─────────────────────────────────────────────────────────────────

# First, we train WITHOUT SMOTE to see the imbalanced baseline.
# This demonstrates why class balancing is needed.

lr_baseline = LogisticRegression(
    max_iter=1000,      # enough iterations to converge
    random_state=42,
    solver='lbfgs'      # efficient solver for small-medium datasets
)

lr_baseline.fit(X_train_scaled, y_train)
y_pred_baseline = lr_baseline.predict(X_test_scaled)
y_prob_baseline = lr_baseline.predict_proba(X_test_scaled)[:, 1]

print("📊 BASELINE MODEL (No SMOTE, Default Threshold=0.5):")
print("="*50)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_baseline):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred_baseline, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred_baseline, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(y_test, y_pred_baseline, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_baseline):.4f}")
print()
print("  Classification Report:")
print(classification_report(y_test, y_pred_baseline, 
      target_names=['Legitimate', 'Fraud'], zero_division=0))

print("""
⚠️  NOTICE: High accuracy but LOW recall for Fraud.
   The model is 'lazy' — it predicts mostly 'Legitimate'
   because 98.5% of training data was legitimate.
   This is WHY we need SMOTE.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 28: Train Improved Model (With SMOTE)
# ─────────────────────────────────────────────────────────────────

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    C=1.0               # Regularization strength (1.0 = default)
)

lr_model.fit(X_train_res, y_train_res)   # ← trained on SMOTE-balanced data

y_pred = lr_model.predict(X_test_scaled)
y_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

print("📊 IMPROVED MODEL (With SMOTE, Default Threshold=0.5):")
print("="*50)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  F1-Score  : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print()
print("  Classification Report:")
print(classification_report(y_test, y_pred, 
      target_names=['Legitimate', 'Fraud'], zero_division=0))

print("""
💡 WITH SMOTE: Recall for Fraud improves significantly.
   We now catch more frauds — which is the PRIMARY goal.
   Trade-off: Precision drops slightly (more false positives),
   but in fraud detection, catching fraud > precision.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 29: Confusion Matrix Visualization
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_p, title in [
    (axes[0], y_pred_baseline, 'Baseline (No SMOTE)'),
    (axes[1], y_pred,          'Improved (With SMOTE)')
]:
    cm = confusion_matrix(y_test, y_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Predicted Legit', 'Predicted Fraud'],
                yticklabels=['Actual Legit',    'Actual Fraud'],
                linewidths=1, linecolor='black', cbar=False,
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(f'Confusion Matrix\n{title}', fontweight='bold', fontsize=13)
    ax.set_ylabel('Actual Label')
    ax.set_xlabel('Predicted Label')
    
    tn, fp, fn, tp = cm.ravel()
    ax.text(0, 2.3, f'TN={tn}  FP={fp}  FN={fn}  TP={tp}', 
            ha='center', fontsize=10, color='gray')

plt.suptitle('Confusion Matrix Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
📖 HOW TO READ THIS:
   TN (True Negative)  = Correctly said 'Legit'  → ✅ Good
   TP (True Positive)  = Correctly said 'Fraud'   → ✅ Great (what we want!)
   FP (False Positive) = Said 'Fraud' but was Legit → 🟡 Annoying for customer
   FN (False Negative) = Said 'Legit' but was Fraud → 🔴 Dangerous! Missed fraud

   In fraud detection: Minimize FN (missed fraud) is the TOP priority.
   SMOTE helps reduce FN by improving Recall.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 30: ROC Curve & Precision-Recall Curve
# ─────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── ROC Curve ─────────────────────────────────────────────────
fpr_b, tpr_b, _ = roc_curve(y_test, y_prob_baseline)
fpr,   tpr,   _ = roc_curve(y_test, y_prob)

auc_b = roc_auc_score(y_test, y_prob_baseline)
auc   = roc_auc_score(y_test, y_prob)

axes[0].plot(fpr_b, tpr_b, color='gray',    lw=2, label=f'Baseline (AUC = {auc_b:.3f})', linestyle='--')
axes[0].plot(fpr,   tpr,   color='#E53935', lw=2, label=f'With SMOTE (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.500)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#E53935')
axes[0].set_title('ROC Curve Comparison', fontweight='bold')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].legend(loc='lower right')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.02])

# ── Precision-Recall Curve ────────────────────────────────────
prec_b, rec_b, _ = precision_recall_curve(y_test, y_prob_baseline)
prec,   rec,   _ = precision_recall_curve(y_test, y_prob)
baseline_pr = y_test.mean()

axes[1].plot(rec_b, prec_b, color='gray',    lw=2, linestyle='--', label='Baseline')
axes[1].plot(rec,   prec,   color='#1565C0', lw=2, label='With SMOTE')
axes[1].axhline(y=baseline_pr, color='black', linestyle=':', label=f'Baseline Rate ({baseline_pr:.3f})')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].set_xlabel('Recall (Fraud Caught)')
axes[1].set_ylabel('Precision (Correctness)')
axes[1].legend()
axes[1].set_xlim([0, 1])

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 INTERPRETATION:
   ROC-AUC Score: Measures model's ability to distinguish fraud from legit.
   → 0.5 = random guessing  |  1.0 = perfect classifier
   → Our model aims for 0.70+ which is good for such severe imbalance.
   
   Precision-Recall Curve: More informative than ROC for imbalanced data.
   Higher curve = better model. The area under this curve is also a key metric.
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 31: Feature Importance (Logistic Regression Coefficients)
# ─────────────────────────────────────────────────────────────────

# In Logistic Regression, larger |coefficient| = more important feature.
# Positive coefficient → increases fraud probability
# Negative coefficient → decreases fraud probability

coef_df = pd.DataFrame({
    'Feature'     : FEATURES,
    'Coefficient' : lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print("📊 LOGISTIC REGRESSION COEFFICIENTS (Importance Ranking):")
print("-"*55)
print(f"  {'Feature':<30} {'Coefficient':>12}  Direction")
print("-"*55)
for _, row in coef_df.iterrows():
    direction = '↑ Increases Fraud Risk' if row['Coefficient'] > 0 else '↓ Decreases Fraud Risk'
    print(f"  {row['Feature']:<30} {row['Coefficient']:>+12.4f}  {direction}")

# Visualization
plt.figure(figsize=(10, 7))
colors = ['#E53935' if c > 0 else '#1565C0' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='black')
plt.axvline(x=0, color='black', linewidth=1.5, linestyle='-')
plt.title('Logistic Regression Feature Coefficients\n(Red = Increases Fraud Risk, Blue = Decreases)', 
          fontweight='bold')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('feature_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()


### 📝 Phase 6 Summary

| Metric | Baseline (No SMOTE) | With SMOTE |
|---|---|---|
| Accuracy | ~98% | Lower (expected) |
| Precision (Fraud) | High | Moderate |
| **Recall (Fraud)** | **Very Low** | **Improved** |
| F1-Score (Fraud) | Low | Better |
| ROC-AUC | Moderate | Higher |

**Key Lesson:** Accuracy is misleading for imbalanced data.  
**Recall** (fraction of actual frauds we caught) is our primary metric.


---
## 🚀 Phase 7 — Model Improvement <a id='phase7'></a>

### Three Improvement Strategies:
1. **Threshold Tuning** — Instead of always using 0.5, find the optimal cutoff
2. **Cross-Validation** — More robust evaluation using K-Fold
3. **Hyperparameter Tuning** — Find the best regularization parameter C


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 32: Threshold Tuning
# ─────────────────────────────────────────────────────────────────

# By default, Logistic Regression uses 0.5 as the cutoff:
# P(fraud) > 0.5 → predict fraud
#
# In fraud detection, we might prefer LOWER threshold (e.g., 0.3):
# → More sensitive → catch more fraud → fewer missed frauds
# Trade-off: More false positives (more legitimate transactions flagged)

thresholds  = np.arange(0.1, 0.9, 0.05)
recalls     = []
precisions  = []
f1_scores   = []

for thresh in thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    recalls.append(recall_score(y_test, y_pred_thresh, zero_division=0))
    precisions.append(precision_score(y_test, y_pred_thresh, zero_division=0))
    f1_scores.append(f1_score(y_test, y_pred_thresh, zero_division=0))

# Find optimal threshold (maximizes F1)
best_idx   = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Metrics vs Threshold ──────────────────────────────────────
axes[0].plot(thresholds, recalls,    'b-o', markersize=4, label='Recall',    linewidth=2)
axes[0].plot(thresholds, precisions, 'r-s', markersize=4, label='Precision', linewidth=2)
axes[0].plot(thresholds, f1_scores,  'g-^', markersize=4, label='F1-Score',  linewidth=2)
axes[0].axvline(best_thresh, color='black', linestyle='--', linewidth=1.5,
                label=f'Best Threshold = {best_thresh:.2f}')
axes[0].set_title('Metrics vs Decision Threshold', fontweight='bold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Score')
axes[0].legend()
axes[0].set_xlim([0.1, 0.9])

# ── F1 Score vs Threshold ─────────────────────────────────────
axes[1].plot(thresholds, f1_scores, 'g-o', markersize=5, linewidth=2)
axes[1].axvline(best_thresh, color='red', linestyle='--', linewidth=2,
                label=f'Optimal: {best_thresh:.2f} (F1={f1_scores[best_idx]:.4f})')
axes[1].set_title('F1-Score vs Decision Threshold', fontweight='bold')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1-Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

# Apply optimal threshold
y_pred_tuned = (y_prob >= best_thresh).astype(int)

print(f"✅ THRESHOLD TUNING RESULTS:")
print(f"   Optimal Threshold : {best_thresh:.2f}")
print(f"   Recall  (Fraud)   : {recall_score(y_test, y_pred_tuned, zero_division=0):.4f}")
print(f"   Precision (Fraud) : {precision_score(y_test, y_pred_tuned, zero_division=0):.4f}")
print(f"   F1-Score (Fraud)  : {f1_score(y_test, y_pred_tuned, zero_division=0):.4f}")
print(f"   ROC-AUC           : {roc_auc_score(y_test, y_prob):.4f}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 33: Cross-Validation (Stratified K-Fold)
# ─────────────────────────────────────────────────────────────────

# Cross-validation gives a more reliable performance estimate.
# Instead of one train-test split, we do K splits and average results.
# StratifiedKFold preserves class ratio in every fold.

print("🔄 STRATIFIED 5-FOLD CROSS-VALIDATION:")
print("-"*50)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_metrics = {'recall': [], 'precision': [], 'f1': [], 'roc_auc': []}

from imblearn.pipeline import Pipeline as ImbPipeline

# Build a pipeline: SMOTE + scaling + model (prevents data leakage in CV)
cv_pipeline = ImbPipeline([
    ('smote',  SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('model',  LogisticRegression(max_iter=1000, random_state=42))
])

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    cv_pipeline.fit(X_tr, y_tr)
    y_val_pred = cv_pipeline.predict(X_val)
    y_val_prob = cv_pipeline.predict_proba(X_val)[:, 1]
    
    cv_metrics['recall'].append(recall_score(y_val, y_val_pred, zero_division=0))
    cv_metrics['precision'].append(precision_score(y_val, y_val_pred, zero_division=0))
    cv_metrics['f1'].append(f1_score(y_val, y_val_pred, zero_division=0))
    cv_metrics['roc_auc'].append(roc_auc_score(y_val, y_val_prob))
    
    print(f"   Fold {fold}: Recall={cv_metrics['recall'][-1]:.4f} | "          f"Precision={cv_metrics['precision'][-1]:.4f} | "          f"F1={cv_metrics['f1'][-1]:.4f} | AUC={cv_metrics['roc_auc'][-1]:.4f}")

print()
for metric, values in cv_metrics.items():
    print(f"   {metric.upper():<12}: {np.mean(values):.4f} ± {np.std(values):.4f}")

print("\n✅ Cross-validation gives a more honest, generalizable performance estimate.")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 34: Hyperparameter Tuning — Regularization Strength C
# ─────────────────────────────────────────────────────────────────

# C = Regularization parameter in Logistic Regression
# Small C  → Strong regularization → Simpler model → May underfit
# Large C  → Weak regularization  → Complex model → May overfit
#
# We try several values and compare F1 scores.

C_values   = [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
c_f1_scores = []

print("🔧 REGULARIZATION TUNING (C parameter):")
print("-"*50)
print(f"  {'C Value':<12} {'Recall':>8} {'Precision':>10} {'F1-Score':>10} {'ROC-AUC':>10}")
print("-"*50)

for C in C_values:
    lr_c = LogisticRegression(max_iter=1000, random_state=42, C=C, solver='lbfgs')
    lr_c.fit(X_train_res, y_train_res)
    yp = lr_c.predict(X_test_scaled)
    yprob_c = lr_c.predict_proba(X_test_scaled)[:, 1]
    
    rec  = recall_score(y_test, yp, zero_division=0)
    prec = precision_score(y_test, yp, zero_division=0)
    f1   = f1_score(y_test, yp, zero_division=0)
    auc  = roc_auc_score(y_test, yprob_c)
    c_f1_scores.append(f1)
    
    marker = ' ← Best' if f1 == max(c_f1_scores) else ''
    print(f"  {C:<12} {rec:>8.4f} {prec:>10.4f} {f1:>10.4f} {auc:>10.4f}{marker}")

best_C = C_values[np.argmax(c_f1_scores)]
print(f"\n✅ Best C value: {best_C}  (F1 = {max(c_f1_scores):.4f})")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 35: Final Best Model
# ─────────────────────────────────────────────────────────────────

# Retrain with best hyperparameters
final_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=best_C,
    solver='lbfgs'
)
final_model.fit(X_train_res, y_train_res)

y_pred_final = (final_model.predict_proba(X_test_scaled)[:, 1] >= best_thresh).astype(int)
y_prob_final = final_model.predict_proba(X_test_scaled)[:, 1]

print("🏆 FINAL MODEL PERFORMANCE SUMMARY:")
print("="*60)
print(f"   Model           : Logistic Regression")
print(f"   C (Regularizer) : {best_C}")
print(f"   Threshold       : {best_thresh:.2f}")
print(f"   Class Balancing : SMOTE")
print("="*60)
print(f"   Accuracy        : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"   Precision (Fraud): {precision_score(y_test, y_pred_final, zero_division=0):.4f}")
print(f"   Recall (Fraud)  : {recall_score(y_test, y_pred_final, zero_division=0):.4f}")
print(f"   F1-Score (Fraud): {f1_score(y_test, y_pred_final, zero_division=0):.4f}")
print(f"   ROC-AUC         : {roc_auc_score(y_test, y_prob_final):.4f}")
print()
print(classification_report(y_test, y_pred_final, 
      target_names=['Legitimate', 'Fraud'], zero_division=0))


---
## 🎯 Phase 8 — Final Insights & Conclusion <a id='phase8'></a>


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 36: Complete Model Comparison Dashboard
# ─────────────────────────────────────────────────────────────────

models = ['Baseline\n(No SMOTE)', 'With SMOTE\n(Default 0.5)', 'Tuned\n(Optimal Threshold)']

recall_scores    = [
    recall_score(y_test, y_pred_baseline, zero_division=0),
    recall_score(y_test, y_pred, zero_division=0),
    recall_score(y_test, y_pred_final, zero_division=0)
]
precision_scores = [
    precision_score(y_test, y_pred_baseline, zero_division=0),
    precision_score(y_test, y_pred, zero_division=0),
    precision_score(y_test, y_pred_final, zero_division=0)
]
f1_scores_all = [
    f1_score(y_test, y_pred_baseline, zero_division=0),
    f1_score(y_test, y_pred, zero_division=0),
    f1_score(y_test, y_pred_final, zero_division=0)
]
auc_scores = [
    roc_auc_score(y_test, y_prob_baseline),
    roc_auc_score(y_test, y_prob),
    roc_auc_score(y_test, y_prob_final)
]

x     = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - 1.5*width, recall_scores,    width, label='Recall',    color='#E53935', edgecolor='black')
ax.bar(x - 0.5*width, precision_scores, width, label='Precision', color='#1565C0', edgecolor='black')
ax.bar(x + 0.5*width, f1_scores_all,    width, label='F1-Score',  color='#2E7D32', edgecolor='black')
ax.bar(x + 1.5*width, auc_scores,       width, label='ROC-AUC',   color='#F57F17', edgecolor='black')

ax.set_title('Model Performance Comparison Dashboard', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right')
ax.axhline(y=1.0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

# Annotate bars
for bar_group in [ax.patches]:
    for bar in bar_group:
        if bar.get_height() > 0.02:
            ax.annotate(f'{bar.get_height():.3f}',
                       xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                       xytext=(0, 3), textcoords='offset points',
                       ha='center', va='bottom', fontsize=8, rotation=90)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 37: Business Insights & Fraud Patterns
# ─────────────────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║           FINAL BUSINESS INSIGHTS & FRAUD PATTERNS          ║
╚══════════════════════════════════════════════════════════════╝

📊 KEY FRAUD PATTERNS DISCOVERED:
─────────────────────────────────────────────────────────────
1. FOREIGN TRANSACTIONS
   → Transactions made abroad are significantly more likely to
     be fraudulent. Banks should require additional verification
     for international transactions.

2. LOCATION MISMATCH
   → When billing address doesn't match transaction location,
     fraud risk spikes dramatically. This is a top fraud signal.

3. HIGH TRANSACTION VELOCITY
   → Users making 5+ transactions in 24 hours are suspicious.
     Real users rarely need that many transactions in one day.
     Fraudsters try to use stolen cards quickly before detection.

4. DEVICE TRUST SCORE
   → Low-trust devices (unfamiliar browsers, new devices)
     correlate with fraud. Multi-factor auth on new devices helps.

5. MERCHANT CATEGORY
   → Grocery has the highest fraud rate, possibly because small
     grocery amounts are harder to notice on bank statements.

6. COMBINED RISK FLAGS
   → Foreign + Location Mismatch together = very high risk.
     These combined signals should trigger automatic review.

💡 MODEL PERFORMANCE SUMMARY:
─────────────────────────────────────────────────────────────
   • The baseline model was nearly useless for fraud detection
     despite high accuracy — proving accuracy is misleading.
   • SMOTE dramatically improved Recall (fraud detection rate).
   • Threshold tuning allowed us to prioritize catching fraud
     over minimizing false alarms.

✅ MODEL STRENGTHS:
─────────────────────────────────────────────────────────────
   • Highly interpretable — can explain to bank management
   • Fast to train and predict — suitable for real-time scoring
   • Probabilistic output — fraud risk score (0–100%) per transaction
   • Low resource requirements — runs on basic hardware

⚠️  MODEL WEAKNESSES:
─────────────────────────────────────────────────────────────
   • Assumes linear decision boundary — may miss complex patterns
   • Sensitive to feature scale (mitigated with StandardScaler)
   • May not capture interaction effects between features
   • Performance may degrade on future fraud patterns (model drift)

🚀 RECOMMENDED NEXT STEPS:
─────────────────────────────────────────────────────────────
   1. Try Random Forest / XGBoost (better at non-linear patterns)
   2. Add more features: IP geolocation, time since last transaction
   3. Deploy as a real-time API scoring service
   4. Set up model monitoring and drift detection
   5. Retrain monthly with new fraud patterns
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 38: Additional ML Models to Try (Recommendations)
# ─────────────────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║           RECOMMENDED NEXT MODELS TO TRY                    ║
╚══════════════════════════════════════════════════════════════╝

🌲 RANDOM FOREST CLASSIFIER
─────────────────────────────────────────────────────────────
   Why: Handles non-linear relationships, robust to outliers.
   How: Trains 100+ decision trees and votes.
   Expected: Better Recall than Logistic Regression.
   
   from sklearn.ensemble import RandomForestClassifier
   rf = RandomForestClassifier(n_estimators=200, class_weight='balanced')

⚡ XGBOOST / LIGHTGBM (Gradient Boosting)
─────────────────────────────────────────────────────────────
   Why: Industry standard for tabular data fraud detection.
   How: Sequentially builds trees, each fixing previous errors.
   Expected: Best overall performance.
   
   import xgboost as xgb
   xgb_model = xgb.XGBClassifier(scale_pos_weight=65, eval_metric='aucpr')

🧠 NEURAL NETWORK (MLP)
─────────────────────────────────────────────────────────────
   Why: Can learn complex interaction patterns.
   How: Multiple layers of weighted connections.
   Expected: Good if dataset grows to 100k+ rows.
   
   from sklearn.neural_network import MLPClassifier
   mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500)

🎯 ISOLATION FOREST (Anomaly Detection)
─────────────────────────────────────────────────────────────
   Why: Treats fraud as anomalies, no labels needed.
   How: Randomly partitions data; anomalies are easier to isolate.
   Expected: Useful when labeled data is scarce.
   
   from sklearn.ensemble import IsolationForest
   iso = IsolationForest(contamination=0.015, random_state=42)

💡 FEATURE ENGINEERING IDEAS:
─────────────────────────────────────────────────────────────
   • amount_log      : log(amount+1) to reduce skewness
   • amount_per_hour : spending rate per hour
   • weekend_flag    : 1 if transaction is on weekend
   • trusted_merchant: 1 if merchant is in user's regular merchants
   • z_score_amount  : how unusual is this amount vs user's history?

🌍 REAL-WORLD APPLICATIONS:
─────────────────────────────────────────────────────────────
   • Bank real-time transaction screening (API endpoint)
   • E-commerce payment gateway fraud filter
   • Insurance claim fraud detection
   • Anti-money laundering (AML) systems
   • Mobile payment fraud scoring (Apple Pay, Google Pay)
   • Buy-Now-Pay-Later (BNPL) fraud detection
""")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 39: Save & Export Artifacts
# ─────────────────────────────────────────────────────────────────

import pickle, os

# Save the trained model
with open('fraud_detection_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)
    
# Save the scaler
with open('fraud_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Artifacts saved:")
print("   fraud_detection_model.pkl  — trained Logistic Regression model")
print("   fraud_scaler.pkl           — fitted StandardScaler")

print("""
To load and use in production:
─────────────────────────────────────
  import pickle
  import pandas as pd
  
  # Load
  model  = pickle.load(open('fraud_detection_model.pkl', 'rb'))
  scaler = pickle.load(open('fraud_scaler.pkl',          'rb'))
  
  # Score a new transaction
  new_transaction = pd.DataFrame([{
      'amount': 450.00, 'transaction_hour': 2,
      'foreign_transaction': 1, 'location_mismatch': 1,
      'device_trust_score': 30, 'velocity_last_24h': 6,
      'cardholder_age': 35, 'combined_risk_flag': 1,
      'high_velocity': 1, 'low_trust_device': 1,
      'night_transaction': 1,
      'merchant_category_Clothing': 0,
      'merchant_category_Electronics': 0,
      'merchant_category_Food': 0,
      'merchant_category_Travel': 1
  }])
  
  scaled_txn   = scaler.transform(new_transaction)
  fraud_prob   = model.predict_proba(scaled_txn)[0][1]
  is_fraud     = fraud_prob >= {best_thresh:.2f}
  
  print(f"Fraud Probability : {{fraud_prob:.2%}}")
  print(f"Decision          : {{'🚨 FRAUD' if is_fraud else '✅ LEGITIMATE'}}")
""".format(best_thresh=best_thresh))


---
## 🏁 Project Complete!

### What You Built:
A **production-quality credit card fraud detection system** including:
- Complete exploratory data analysis with 10+ professional visualizations
- Feature engineering with domain-knowledge-based risk flags
- Proper handling of severe class imbalance using SMOTE
- Logistic Regression model with threshold tuning and hyperparameter optimization
- Full evaluation using Recall, Precision, F1-Score, ROC-AUC, and Confusion Matrix
- Saved model artifacts ready for deployment

### What You Learned:
1. Why **accuracy is misleading** for imbalanced datasets
2. How **SMOTE** creates synthetic minority samples to fix class imbalance
3. Why **threshold tuning** matters — 0.5 is not always the best cutoff
4. How to read **Confusion Matrices**, **ROC curves**, and **Precision-Recall curves**
5. How **Logistic Regression coefficients** tell us which features drive fraud
6. How to structure a **production ML pipeline** properly

---
*Generated with ❤️ — Professional ML Pipeline for Credit Card Fraud Detection*
